# SQLarper's ETL Pipeline

## 1. Configuration
\
Everything a rerun might need to change lives in this one cell.
- `TARGET` picks the database: `"TEST"` while building, `"PROD"` only for the final load. That's the only line you change.
- Passwords are not here. They're read from `.env` (`DEV_DB_SERVER` for TEST, `PROD_DB_URL` for PROD).
- `DATE_FOUND` is the day `scraping.py` downloaded the raw files. It's stored in `event_source.date_found`.


Holds everything that a rerun might need to change.

In [2]:
import html
import json
import os
import random
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import polars as pl
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text

# ---- Which database to load into: "TEST" or "PROD" ----
TARGET = "TEST"
TEST_DB_NAME = "_catcruxis"

# ---- Raw data files (downloaded by scraping.py) ----
RAW_DIR = Path("../data/raw")
WILDCATSYNC_FILE = RAW_DIR / "wildcatsync.xml"
HURT_HUB_FILE = RAW_DIR / "hurthub.json"
DAVIDSON_FILE = RAW_DIR / "davidson.json"

# davidson.edu gives local times without a time zone
LOCAL_TZ = "America/New_York"

# When scraping.py downloaded the raw files. Update this if you rerun scraping.py.
DATE_FOUND = datetime(2026, 11, 4, tzinfo=timezone.utc)


# ---- Connect ----
load_dotenv(find_dotenv(usecwd=True))   # walks up the folders until it finds .env

if TARGET == "TEST":
    db_url = f"{os.environ['DEV_DB_SERVER']}/{TEST_DB_NAME}"
elif TARGET == "PROD":
    db_url = os.environ["PROD_DB_URL"]
else:
    raise ValueError("TARGET must be 'TEST' or 'PROD'")

db_url = db_url.replace("postgres://", "postgresql://", 1)
engine = create_engine(db_url)

print("Polars", pl.__version__)
print("Loading into:", TARGET)   # never print db_url, it contains your password

Polars 1.42.1
Loading into: TEST


## 2. Extract

We read each source exactly as it arrives. Nothing gets fixed here; every fix is in Transform, as code.

| Source | File | Retrieved from |
|---|---|---|
| WildcatSync | `data/raw/wildcatsync.xml` | https://wildcatsync.davidson.edu/events.rss |
| Hurt Hub | `data/raw/hurthub.json` | https://hurthub.davidson.edu/wp-json/tribe/events/v1/events |
| Davidson College Events | `data/raw/davidson.json` | https://www.davidson.edu/events (scraped) |

`scraping.py` downloads all three. To refresh the data, run `python scraping.py`, then rerun this notebook.

### 2a. WildcatSync (RSS/XML) plus one line: “Each <item> is one event. Some fields are in an XML namespace called events. An event can have several <host> tags, so hosts stay a list for now.”

In [3]:
EV = "{events}"   # the namespace WildcatSync uses for start, end, location, status, host

items = ET.parse(WILDCATSYNC_FILE).getroot().findall(".//item")

ws_raw = pl.DataFrame(
    [
        {
            "title": it.findtext("title"),
            "link": it.findtext("link"),
            "description_html": it.findtext("description"),
            "category": it.findtext("category"),
            "start": it.findtext(EV + "start"),
            "end": it.findtext(EV + "end"),
            "location": it.findtext(EV + "location"),
            "status": it.findtext(EV + "status"),
            "hosts": [h.text for h in it.findall(EV + "host")],
        }
        for it in items
    ],
    schema_overrides={"hosts": pl.List(pl.String)},
)
print(ws_raw.shape)
ws_raw.head(3)

(106, 9)


title,link,description_html,category,start,end,location,status,hosts
str,str,str,str,str,str,str,str,list[str]
"""Social Gathering ""","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""Social""","""Wed, 04 Nov 2026 23:30:00 GMT""","""Thu, 05 Nov 2026 00:30:00 GMT""","""Watson 2nd Floor Lounge""","""confirmed""","[""Davidson College ACM Student Chapter""]"
"""Wednesday Night Zen""","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""Spirituality""","""Thu, 05 Nov 2026 00:00:00 GMT""","""Thu, 05 Nov 2026 01:00:00 GMT""","""Oasis""","""confirmed""","[""Buddhism at Davidson""]"
"""Mewing, Mogging, and Mental He…","""https://wildcatsync.davidson.e…","""<div class=""h-event vevent""> …","""ThoughtfulLearning""","""Thu, 05 Nov 2026 00:00:00 GMT""","""Thu, 05 Nov 2026 01:00:00 GMT""","""900 Room""","""confirmed""","[""Center for Student Health & Well-Being""]"


### 2b. Hurt Hub (JSON API) plus: “venue is an object for in-person events and an empty list for virtual ones. organizer and categories are lists of objects; we keep only the names.”

In [4]:
hh_json = json.loads(HURT_HUB_FILE.read_text(encoding="utf-8"))["events"]

hh_raw = pl.DataFrame(
    [
        {
            "title": e["title"],
            "url": e["url"],
            "description_html": e["description"],
            "utc_start_date": e["utc_start_date"],
            "utc_end_date": e["utc_end_date"],
            "venue": e["venue"].get("venue") if isinstance(e["venue"], dict) else None,
            "organizers": [o["organizer"] for o in e["organizer"]],
            "categories": [c["name"] for c in e["categories"]],
        }
        for e in hh_json
    ],
    schema_overrides={"organizers": pl.List(pl.String), "categories": pl.List(pl.String)},
)
print(hh_raw.shape)
hh_raw

(6, 8)


title,url,description_html,utc_start_date,utc_end_date,venue,organizers,categories
str,str,str,str,str,str,list[str],list[str]
"""Building a Lean Startup &#8211…","""https://hurthub.davidson.edu/e…","""<div>Learn how to jumpstart yo…","""2026-09-23 22:45:00""","""2026-11-12 01:15:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]","[""Class"", ""Discussion""]"
"""The Extended Mind: A Field Gui…","""https://hurthub.davidson.edu/e…","""<div>Facilitated by Peter Gray…","""2026-11-04 21:30:00""","""2026-11-04 22:30:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]
"""Shop Local at the Makers&#8217…","""https://hurthub.davidson.edu/e…","""<p>The holidays are just aroun…","""2026-11-13 21:00:00""","""2026-11-14 00:00:00""","""The Hurt Hub at Davidson Colle…",[],"[""Social""]"
"""The Hurt Hub Innovation &amp; …","""https://hurthub.davidson.edu/e…","""<div>The Hurt Hub Innovation &…","""2026-11-18 23:00:00""","""2026-11-19 02:00:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]
"""CPA Office Hours (SCHWINN CPA …","""https://hurthub.davidson.edu/e…","""<p>The Hurt Hub has partnered …","""2026-11-19 14:30:00""","""2026-11-19 17:30:00""",null,[],"[""Amplify Davidson"", ""Discussion"", … ""Virtual""]"
"""Solopreneur Circle""","""https://hurthub.davidson.edu/e…","""<div>A monthly community gathe…","""2026-11-20 14:00:00""","""2026-11-20 15:00:00""","""The Hurt Hub at Davidson Colle…","[""The Hurt Hub@Davidson""]",[]


### 2c. Davidson College Events (scraped JSON) plus: “scraping.py already turned each event page into one object. start and end are local Davidson times with no time zone.”

In [5]:
dv_json = json.loads(DAVIDSON_FILE.read_text(encoding="utf-8"))["events"]

dv_raw = pl.DataFrame(
    [
        {k: e.get(k) for k in ("title", "url", "description", "start", "end", "location", "date_time_text")}
        for e in dv_json
    ]
)
print(dv_raw.shape)
dv_raw.head(3)

(45, 7)


title,url,description,start,end,location,date_time_text
str,str,str,str,str,str,str
"""Charif Shanahan Poetry Reading""","""https://www.davidson.edu/event…","""November 04 5:00PM - 6:30PM …","""2026-11-04T17:00:00""","""2026-11-04T18:30:00""","""UNION 307 - Smith 900 Room""","""November 04 5:00PM - 6:30PM"""
"""Solar Canopy Dedication""","""https://www.davidson.edu/event…","""November 05 11:05AM - 12:00PM…","""2026-11-05T11:05:00""","""2026-11-05T12:00:00""","""Baker Sports Complex South Law…","""November 05 11:05AM - 12:00PM"""
"""Piano Studio Fall Recital""","""https://www.davidson.edu/event…","""November 05 7:30PM - 9:00PM …","""2026-11-05T19:30:00""","""2026-11-05T21:00:00""","""SLOAN TY-TL - Tyler-Tallman Re…","""November 05 7:30PM - 9:00PM"""


### 2d. Profiling
Before fixing anything, we look at what we have: missing values, multi-valued cells, formats and repeats. Each finding becomes a cleaning step in section 3 (labelled F1, F2, ...).

In [6]:
for name, df in (("WildcatSync", ws_raw), ("Hurt Hub", hh_raw), ("Davidson", dv_raw)):
    print(name, df.null_count().to_dicts()[0])

WildcatSync {'title': 0, 'link': 0, 'description_html': 0, 'category': 0, 'start': 0, 'end': 0, 'location': 0, 'status': 0, 'hosts': 0}
Hurt Hub {'title': 0, 'url': 0, 'description_html': 0, 'utc_start_date': 0, 'utc_end_date': 0, 'venue': 1, 'organizers': 0, 'categories': 0}
Davidson {'title': 0, 'url': 0, 'description': 0, 'start': 0, 'end': 0, 'location': 0, 'date_time_text': 0}


Cancelled events and hosts per event:



In [7]:
print(ws_raw["status"].value_counts())
print(ws_raw.select(pl.col("hosts").list.len().alias("hosts_per_event"))["hosts_per_event"].value_counts().sort("hosts_per_event"))

shape: (2, 2)
┌───────────┬───────┐
│ status    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ confirmed ┆ 104   │
│ cancelled ┆ 2     │
└───────────┴───────┘
shape: (6, 2)
┌─────────────────┬───────┐
│ hosts_per_event ┆ count │
│ ---             ┆ ---   │
│ u32             ┆ u32   │
╞═════════════════╪═══════╡
│ 1               ┆ 61    │
│ 2               ┆ 36    │
│ 3               ┆ 4     │
│ 4               ┆ 2     │
│ 5               ┆ 2     │
│ 6               ┆ 1     │
└─────────────────┴───────┘


Categories:


In [8]:
print(ws_raw["category"].value_counts().sort("count", descending=True))
print(hh_raw.select(pl.col("categories").explode())["categories"].value_counts())

shape: (9, 2)
┌────────────────────┬───────┐
│ category           ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ ThoughtfulLearning ┆ 42    │
│ Spirituality       ┆ 16    │
│ Social             ┆ 15    │
│ Athletics          ┆ 15    │
│ Cultural           ┆ 8     │
│ CommunityService   ┆ 4     │
│ Fundraising        ┆ 3     │
│ Arts               ┆ 2     │
│ GroupBusiness      ┆ 1     │
└────────────────────┴───────┘
shape: (8, 2)
┌───────────────────┬───────┐
│ categories        ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ null              ┆ 3     │
│ Virtual           ┆ 1     │
│ Class             ┆ 1     │
│ Discussion        ┆ 2     │
│ Office Hours      ┆ 1     │
│ Amplify Davidson  ┆ 1     │
│ Social            ┆ 1     │
│ Free for Students ┆ 1     │
└───────────────────┴───────┘


/tmp/ipykernel_1735507/1681116005.py:2: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  print(hh_raw.select(pl.col("categories").explode())["categories"].value_counts())


Date formats:


In [9]:
print("WildcatSync:", ws_raw["start"][0])
print("Hurt Hub:   ", hh_raw["utc_start_date"][0])
print("Davidson:   ", dv_raw["start"][0])

WildcatSync: Wed, 04 Nov 2026 23:30:00 GMT
Hurt Hub:    2026-09-23 22:45:00
Davidson:    2026-11-04T17:00:00


Location text:



In [10]:
all_locations = pl.concat([
    ws_raw.select("location"),
    hh_raw.select(pl.col("venue").alias("location")),
    dv_raw.select("location"),
])
print("distinct location strings:", all_locations["location"].n_unique())
all_locations["location"].value_counts().sort("count", descending=True).head(25)

distinct location strings: 81


location,count
str,u32
"""SLOAN TY-TL - Tyler-Tallman Re…",19
"""Cannon Pool - Baker Sports Com…",11
"""Duke Family Performance Hall""",8
"""Oasis""",8
"""The Hurt Hub at Davidson Colle…",5
…,…
"""Norwood""",2
"""DCPC Sanctuary""",2
"""WALL Hamilton W. McKay Atrium""",1


Messy titles:



In [11]:
print(ws_raw.filter(pl.col("title") != pl.col("title").str.strip_chars()).select("title").head())
print(hh_raw.filter(pl.col("title").str.contains("&")).select("title"))

shape: (5, 1)
┌─────────────────────────────────┐
│ title                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Social Gathering                │
│ Mindful Yoga                    │
│ A Conversation About Vaccines … │
│ Cats Excursions: Murder on the… │
│  Yearbook Meeting  ✏️           │
└─────────────────────────────────┘
shape: (3, 1)
┌─────────────────────────────────┐
│ title                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Building a Lean Startup &#8211… │
│ Shop Local at the Makers&#8217… │
│ The Hurt Hub Innovation &amp; … │
└─────────────────────────────────┘


The same event on two sources, or posted twice:



In [12]:
def title_key(col):
    return pl.col(col).str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")

ws_keys = ws_raw.select(title_key("title").alias("k"), "start")
dv_keys = dv_raw.select(title_key("title").alias("k"))
print("on both WildcatSync and davidson.edu:")
print(ws_keys.join(dv_keys, on="k", how="inner").select("k").unique())
print("posted twice on WildcatSync (same title and start):")
print(ws_keys.filter(pl.struct("k", "start").is_duplicated()))

on both WildcatSync and davidson.edu:
shape: (2, 1)
┌─────────────────────────────────┐
│ k                               │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ solarcanopydedication           │
│ remembranceandreconnectionacom… │
└─────────────────────────────────┘
posted twice on WildcatSync (same title and start):
shape: (4, 2)
┌─────────────────────────────┬───────────────────────────────┐
│ k                           ┆ start                         │
│ ---                         ┆ ---                           │
│ str                         ┆ str                           │
╞═════════════════════════════╪═══════════════════════════════╡
│ bonnerscholarsmondaymeeting ┆ Mon, 16 Nov 2026 22:00:00 GMT │
│ bonnerscholarsmondaymeeting ┆ Mon, 16 Nov 2026 22:00:00 GMT │
│ largegroup                  ┆ Wed, 18 Nov 2026 00:30:00 GMT │
│ largegroup                  ┆ Wed, 18 Nov 2026 00:30:00 GMT │
└─────────────────

## 3. Transform: clean

## 4. Transform: normalize

## 5. Load